# DSE Baselines v2 — Notebook 2: PyTorch Geometric & Transformer (`DSGAT`, `A3Net`, `HSTrans`)

Các sửa đổi trọng yếu trong Protocol v2:
- **DSGAT & A3Net**:
  - Ở **Split B (Cold-start)**, chỉ đưa 540 thuốc `train_drugs` vào `train_loader` (đúng thiết kế `ICS.py` / `cold-scence.py` của tác giả gốc, thay vì để thuốc cold-start bị phạt về `eps = 0.5`).
  - Dựng đồ thị k-NN của SE theo từng fold từ `scipy.sparse` (không gọi `nx.from_numpy_matrix` đã bị xóa trên NetworkX 3.x).
- **HSTrans**:
  - Trích xuất top-50 BPE substructures (`identify_sub`) **theo từng fold chỉ từ tập `train` (80%)** đã vector hóa bằng NumPy (<0.05s/fold), khử hoàn toàn rò rỉ nhãn test.
  - Tiền mã hóa BPE cho 750 thuốc 1 lần trên RAM (tăng tốc ~50× so với đọc CSV trong `__getitem__`).

In [ ]:
# ===== 1. CLONE / UPDATE REPO & MOUNT GOOGLE DRIVE =====
import os, subprocess, torch

REPO_URL = 'https://github.com/hungbuile04/DSE.git'
REPO_BRANCH = 'main'
WORK_DIR = '/content/DSE'

if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=False)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

# Mount Google Drive for per-fold auto-backup & seamless resume
DRIVE_DIR = '/content/drive/MyDrive/DSE_results_v2'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted. Auto-sync directory: {DRIVE_DIR}')
except Exception as e:
    DRIVE_DIR = '/content/DSE_drive_fallback'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive not mounted ({e}); using local fallback: {DRIVE_DIR}')

os.environ['DRIVE_DIR'] = DRIVE_DIR

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('medium')
    except Exception:
        pass
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: CUDA GPU not detected! Enable Runtime -> Change runtime type -> GPU.')

# Install PyTorch Geometric, RDKit, and subword-nmt (pure wheel install, works across all Colab PyTorch 2.x builds)
!pip install -q torch_geometric rdkit subword-nmt scipy pandas scikit-learn
import torch_geometric, rdkit
print('PyG version:', torch_geometric.__version__, '| RDKit:', rdkit.__version__)

In [ ]:
# ===== 2. VERIFY BENCHMARK DATA INTEGRITY (SHA256) =====
!python MSSF_plus/audit/verify_checksums.py --check

---
## 3. Baseline 4: `DSGAT` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_dsgat_a3net \
    --model DSGAT \
    --split B \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

!python -m MSSF_plus.baselines_rerun.run_dsgat_a3net \
    --model DSGAT \
    --split A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 4. Baseline 5: `A3Net` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_dsgat_a3net \
    --model A3Net \
    --split B \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

!python -m MSSF_plus.baselines_rerun.run_dsgat_a3net \
    --model A3Net \
    --split A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 5. Baseline 6: `HSTrans` (Split B & Split A — 10 Folds)

In [ ]:
%%time
!python -m MSSF_plus.baselines_rerun.run_hstrans \
    --split B \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

!python -m MSSF_plus.baselines_rerun.run_hstrans \
    --split A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 6. Kiểm tra kết quả trên Google Drive

In [ ]:
!python -m MSSF_plus.scripts.summarize_results --results_dir "$DRIVE_DIR" --ref_method MSSF_v2
!python shared_data/unified_eval.py --results_dir "$DRIVE_DIR" 